In [1]:
from google.colab import drive, files

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
uploaded  = files.upload()

Saving lgbm_avoidable_er_model.pkl to lgbm_avoidable_er_model.pkl
Saving X_test_indices.npy to X_test_indices.npy
Saving X_train_indices.npy to X_train_indices.npy
Saving y_test_indices.npy to y_test_indices.npy
Saving member_features.parquet to member_features.parquet
Saving members_phase1 (1).parquet to members_phase1 (1).parquet
Saving claims_phase1.parquet to claims_phase1.parquet
Saving providers_phase1.parquet to providers_phase1.parquet


In [4]:
import pandas as pd

In [5]:
member_features = pd.read_parquet ("member_features.parquet")

In [7]:
claims= pd.read_parquet ("claims_phase1.parquet")
members = pd.read_parquet ("members_phase1 (1).parquet")
providers = pd.read_parquet ("providers_phase1.parquet")

In [8]:
member_features.columns

Index(['member_id', 'num_chronic_conditions', 'diabetes_flag', 'chf_flag',
       'copd_flag', 'renal_flag', 'behavioral_health_flag',
       'prior_er_visits_90d', 'pcp_touch_90d', 'risk_bin', 'has_avoidable_er'],
      dtype='object')

In [9]:
claims.columns

Index(['member_id', 'age', 'gender', 'region', 'chronic_count',
       'behavioral_flag', 'risk_level', 'annual_claims', 'claim_id', 'date',
       'place_of_service', 'diagnosis_group', 'admission_flag',
       'allowed_amount', 'avoidable_er', 'provider_id', 'speciality',
       'provider_region', 'pcp_engaged', 'region_multiplier'],
      dtype='object')

In [10]:
members.columns

Index(['member_id', 'age', 'gender', 'region', 'chronic_count',
       'behavioral_flag', 'risk_level', 'annual_claims', 'pcp_engaged',
       'diabetes_flag', 'chf_flag', 'copd_flag', 'renal_flag'],
      dtype='object')

In [11]:
providers.columns

Index(['provider_id', 'speciality', 'provider_region'], dtype='object')

In [12]:
claims["age_bin"] = pd.cut(
    claims["age"],
    bins =[0,18,35,50,65,80,100],
    labels=["0-18", "18-35", "35-50", "50-65", "65-80", "80+"]
)

In [13]:
quasi_identifiers = ["age_bin", "gender", "region", "diagnosis_group" ]

In [14]:
group_sizes = (
    claims.groupby (quasi_identifiers, observed=True)
    .size()
    .reset_index (name= "count")
)

In [15]:
min_k = group_sizes["count"].min()

In [16]:
violations_k5 = (group_sizes["count"]<5).sum()

In [17]:
violations_k10 = (group_sizes["count"]<10).sum()

In [18]:
print ("Minimum k",min_k)
print ("Violations at k=5 : ",violations_k5)
print ("Violations at k=10 : ",violations_k10)

Minimum k 570
Violations at k=5 :  0
Violations at k=10 :  0


In [19]:
claims["risk_level"].unique()

['Medium', 'High', 'Low']
Categories (3, object): ['Low' < 'Medium' < 'High']

On Mitigation

In [20]:
claims["age_bin_privacy"] =  pd.cut(
    claims["age"],
    bins =[0,30,45,60,75,100],
    labels =["0-30","30-45","45-60","60-75","75+"]
)

In [21]:
region_map = {
    "North": "NorthCentral",
    "Central": "NorthCentral",
    "East": "EastWest",
    "West": "EastWest",
    "South": "SouthCostal",
    "Costal": "SouthCostal"
}

In [22]:
claims["region_privacy"]= claims["region"].map(region_map)

In [23]:
quasi_identifiers_privacy = [
    "age_bin_privacy",
    "gender",
    "region_privacy",
    "diagnosis_group"
]

In [24]:
group_sizes_privacy = (
    claims.groupby(quasi_identifiers_privacy, observed= True)
    .size()
    .reset_index(name= "count")
)

In [25]:
new_min_k = group_sizes_privacy["count"].min()

In [26]:
new_k5_violations = (group_sizes_privacy["count"]<5).sum()

In [27]:
new_k10_violations = (group_sizes_privacy["count"]<10).sum()

In [28]:
print ("New Minimum k",new_min_k)
print ("New Violations at k=5 : ",new_k5_violations)
print ("New Violations at k=10 : ",new_k10_violations)

New Minimum k 2643
New Violations at k=5 :  0
New Violations at k=10 :  0


| Configuration | Minimum k |
| ------------- | --------- |
| Original bins | 570       |
| Coarser bins  | 2643      |


In [29]:
l_diversity = (
    claims.groupby(quasi_identifiers, observed= True)["risk_level"]
    .nunique()
    .reset_index(name ="distinct_risk_levels")
)

In [30]:
l_min = l_diversity["distinct_risk_levels"].min()

In [31]:
l_min

3

In [32]:
global_dist =  claims["risk_level"].value_counts(normalize= True)

In [33]:
group_dist = (
    claims.groupby(quasi_identifiers, observed = True)["risk_level"]
    .value_counts(normalize = True)
    .unstack (fill_value=0)
)

In [34]:
t_values = (group_dist - global_dist).abs().max(axis=1)

In [35]:
t_max = t_values.max()

In [36]:
t_max

0.3712068050677795

In [37]:
dominance = (
    claims.groupby(quasi_identifiers, observed=True)["risk_level"]
    .value_counts(normalize=True)
    .groupby(level=0)
    .max()
)

/tmp/ipython-input-3248973463.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(level=0)


In [38]:
max_dominance= dominance.max()

In [39]:
max_dominance

0.7072053311120367

| Metric      | Result | Risk Level                  |
| ----------- | ------ | --------------------------- |
| k-anonymity | 570    | Extremely low identity risk |
| l-diversity | 3      | No homogeneity risk         |
| Dominance   | 0.707  | Moderate but acceptable     |
| t-closeness | 0.371  | Moderate distribution skew  |


In [40]:
member_k = (
    claims.groupby(quasi_identifiers, observed=True)["member_id"]
    .nunique()
    .reset_index(name="unique_members")
)

In [41]:
member_min_k = member_k["unique_members"].min()

In [42]:
member_min_k

186

In [43]:
(member_k["unique_members"] < 10).sum()

np.int64(0)

In [44]:
behavioral_l = (
    claims.groupby(quasi_identifiers, observed=True)["behavioral_flag"]
    .nunique()
    .min()
)

In [45]:
behavioral_l

2

In [46]:
admission_l = (
    claims.groupby(quasi_identifiers, observed=True)["admission_flag"]
    .nunique()
    .min()
)

In [47]:
admission_l

2

| Metric                        | Result | Interpretation                |
| ----------------------------- | ------ | ----------------------------- |
| Claim-level k                 | 570    | Extremely strong              |
| Member-level k                | 186    | Strong person-level anonymity |
| l-diversity (risk_level)      | 3      | Full diversity                |
| l-diversity (behavioral_flag) | 2      | No homogeneity                |
| l-diversity (admission_flag)  | 2      | No homogeneity                |
| Max dominance                 | 0.707  | Acceptable                    |
| t-closeness                   | 0.371  | Moderate but natural skew     |


In [48]:
member_features.index[:20]

RangeIndex(start=0, stop=20, step=1)

In [49]:
uploaded  = files.upload()

In [50]:
import joblib

In [51]:
model = joblib.load("lgbm_avoidable_er_model.pkl")

In [52]:
from sklearn.metrics  import roc_auc_score
import numpy as np

In [53]:
test_idx = np.load("X_test_indices.npy")
train_idx = np.load("X_train_indices.npy")

In [54]:
X= member_features.drop (columns = ["has_avoidable_er", "member_id"])

In [55]:
y= member_features["has_avoidable_er"]

In [56]:
X_test = X.loc [test_idx]
y_test = y.loc[test_idx]

In [57]:
from sklearn.metrics import roc_auc_score

y_pred_proba = model.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, y_pred_proba)


np.float64(0.6376966004736456)

In [58]:
X_test.index

Index([20685,   460, 37495, 27121, 23224, 18280,  7893, 17362, 15052, 31939,
       ...
       17156,  8707, 32554, 31062, 19362, 33171,   828, 31692, 14491, 25094],
      dtype='int64', length=10000)

During model reproducibility validation, a difference was observed between the previously reported AUC (0.687) and the re-estimated AUC (0.638). This variation is attributable to regeneration of the synthetic dataset after introducing additional stochastic cost simulations (np.random.uniform). Although a global random seed was fixed, the insertion of new random draws altered the pseudorandom number stream, resulting in a different synthetic population. Consequently, feature distributions and target realizations shifted slightly, producing a new but internally consistent model performance estimate. Importantly, privacy evaluation did not modify modeling features or target definitions. The observed AUC difference reflects dataset regeneration effects rather than privacy-induced degradation. Model performance remains within the expected behavioral prediction range.

In [71]:
import os
import json
import zipfile
import joblib
import numpy as np
from google.colab import files

print("Saving artifacts...")

member_features.to_parquet("member_features_final.parquet", index=False)

claims.to_parquet("claims_privacy_evaluated.parquet", index=False)

joblib.dump(model, "lgbm_avoidable_er_model_final.pkl")

np.save("X_test_indices_final.npy", test_idx)
np.save("X_train_indices_final.npy", train_idx)

from sklearn.metrics import roc_auc_score

utility_metrics = {
    "auc_test": float(roc_auc_score(y_test, model.predict_proba(X_test)[:,1])),
    "er_rate": float((claims["place_of_service"]=="ER").mean()),
    "risk_distribution": member_features["risk_bin"].value_counts(normalize=True).to_dict()
}

with open("utility_metrics.json", "w") as f:
    json.dump(utility_metrics, f, indent=4)

privacy_metrics = {
    "claim_level_k_min": int(min_k),
    "member_level_k_min": int(member_min_k),
    "l_diversity_risk_level": int(l_min),
    "l_diversity_behavioral_flag": int(behavioral_l),
    "l_diversity_admission_flag": int(admission_l),
    "t_closeness_max": float(t_max),
    "max_dominance": float(max_dominance)
}

with open("privacy_metrics.json", "w") as f:
    json.dump(privacy_metrics, f, indent=4)
artifact_files = [
    "member_features_final.parquet",
    "claims_privacy_evaluated.parquet",
    "lgbm_avoidable_er_model_final.pkl",
    "X_test_indices_final.npy",
    "X_train_indices_final.npy",
    "utility_metrics.json",
    "privacy_metrics.json"
]

with zipfile.ZipFile("phase6_artifacts.zip", "w") as z:
    for file in artifact_files:
        if os.path.exists(file):
            z.write(file)
        else:
            print(f"Warning: {file} not found.")

print("ZIP file created: phase6_artifacts.zip")

files.download("phase6_artifacts.zip")

print("Download triggered successfully.")

Saving artifacts...
ZIP file created: phase6_artifacts.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered successfully.
